In [1]:
import pandas as pd
import numpy as np
import math
import json

In [2]:
# ==============================================================
# Data Initialization
# ==============================================================
cbgs = pd.read_csv("worcester_cbgs.csv")
pois = pd.read_csv("worcester_pois.csv")
visits = pd.read_csv("worcester_cbg_poi_visits.csv")
distances = pd.read_csv("worcester_cbg_poi_distance.csv")
parameters = pd.read_csv("calibrated_parameters_filtered.csv")

with open("worcester_cbgs_map.geojson", "r") as f:
    geojson = json.load(f)
 
centroid_rows = []
for feature in geojson["features"]:
    props = feature["properties"]
    centroid_rows.append({
        "cbg": int(props["GEOID10"]),
        "lat": float(props["INTPTLAT10"]),
        "lon": float(props["INTPTLON10"])
    })
 
cbg_centroids = pd.DataFrame(centroid_rows)
cbgs = cbgs.merge(cbg_centroids, on="cbg", how="left")

print("Files loaded:")
print(f"  CBGs:       {len(cbgs)} neighborhoods")
print(f"  POIs:       {len(pois)} existing businesses")
print(f"  Visits:     {len(visits)} records")
print(f"  Distances:  {len(distances)} records")
print(f"  Categories: {len(parameters)} parameter sets")

Files loaded:
  CBGs:       149 neighborhoods
  POIs:       4069 existing businesses
  Visits:     26924 records
  Distances:  606281 records
  Categories: 23 parameter sets


In [ ]:
from numpy.random import beta
# ==============================================================
# User Input Layer
# ==============================================================
print("=" * 60)
print ("Enter new store details")
print("=" * 60)

print("\nStore coordinates")
latitude_new = float(input("Latitude (e.g., 42.2625): "))
longitude_new = float(input("Longitude (e.g., -71.8032)"))

print("\nAvailable categories: ")
for i, row in parameters.iterrows():
    print(f"  {str(i).rjust(2)}. {row['top_category']}  (NAICS: {row['NAICS code']})")

print("\nEnter business category")
user_input = input("Type exact top_category name or NAICS code: ").strip()

store_size = float(input("\n Store size in square meters (e.g., 2500): "))

# Parameter Lookup
parameters_row = parameters[parameters['top_category'].str.lower() == user_input.lower()]

if parameters_row.empty:
    try:
        naics_int = int(user_input)
        parameters_row = parameters[parameters["NAICS code"] == naics_int]
    except ValueError:
        pass

if parameters_row.empty:
    print(f"\nERROR: Category '{user_input}' not found.")
    print("Re-run and choose an exact name or NAICS code from the list")
    exit()

alpha = parameters_row.iloc[0]["alpha"]
beta = parameters_row.iloc[0]["beta"]
matched_category = parameters_row.iloc[0]["top_category"]
naics_code = parameters_row.iloc[0]["NAICS code"]

print(f"\nMatched: '{matched_category}'")
print(f"Alpha = {alpha} | Beta = {beta}")

Enter new store details

Store coordinates

Available categories: 
   0. Building Material and Supplies Dealers  (NAICS: 4441)
   1. Bakeries and Tortilla Manufacturing  (NAICS: 311811)
   2. Other Miscellaneous Manufacturing  (NAICS: 3399)
   3. Gasoline Stations  (NAICS: 447110)
   4. Offices of Dentists  (NAICS: 621210)
   5. Activities Related to Credit Intermediation  (NAICS: 522310)
   6. Justice, Public Order, and Safety Activities  (NAICS: 922110)
   7. Other Miscellaneous Store Retailers  (NAICS: 453991)
   8. Automotive Parts, Accessories, and Tire Stores  (NAICS: 441310)
   9. Beer, Wine, and Liquor Stores  (NAICS: 445310)
  10. General Merchandise Stores, including Warehouse Clubs and Supercenters  (NAICS: 452319)
  11. Lessors of Real Estate  (NAICS: 531120)
  12. Depository Credit Intermediation  (NAICS: 522110)
  13. Colleges, Universities, and Professional Schools  (NAICS: 611310)
  14. Offices of Real Estate Agents and Brokers  (NAICS: 531210)
  15. Other Financial Inv

IndexError: single positional indexer is out-of-bounds

: 

In [ ]:
# ==============================================================
# Distance Calculation
# Straight-Line Euclidean Distances calculated via a Projected Coordinate System (Latitude/Longitude in WGS84)
# 1. Project lat/lon (WGS84 degrees) into UTM Zone 19N (EPSG:32619) which gives us flat x, y coordinates in METERS
# 2. Apply standard Euclidean distance: sqrt((x2-x1)^2 + (y2-y1)^2)
# ==============================================================
def wgs84_to_utm_zone19n(latitude, longitude):

    a  = 6_378_137.0
    f  = 1 / 298.257223563
    e2 = 2 * f - f ** 2
    k0 = 0.9996
    E0 = 500_000.0

    lon0  = math.radians(-69.0)
    lat_r = math.radians(latitude)
    lon_r = math.radians(longitude)

    N = a / math.sqrt(1 - e2 * math.sin(lat_r) ** 2)
    T = math.tan(lat_r) ** 2
    C = (e2 / (1 - e2)) * math.cos(lat_r) ** 2
    A = math.cos(lat_r) * (lon_r - lon0)

    M = a * (
        (1 - e2/4 - 3*e2**2/64 - 5*e2**3/256)   * lat_r
      - (3*e2/8 + 3*e2**2/32 + 45*e2**3/1024)   * math.sin(2 * lat_r)
      + (15*e2**2/256 + 45*e2**3/1024)           * math.sin(4 * lat_r)
      - (35*e2**3/3072)                           * math.sin(6 * lat_r)
    )

    x = E0 + k0 * N * (
        A
        + (1 - T + C)                                * A**3 / 6
        + (5 - 18*T + T**2 + 72*C - 58*(e2/(1-e2))) * A**5 / 120
    )
    y = k0 * (
        M + N * math.tan(lat_r) * (
            A**2 / 2
            + (5 - T + 9*C + 4*C**2)                        * A**4 / 24
            + (61 - 58*T + T**2 + 600*C - 330*(e2/(1-e2)))  * A**6 / 720
        )
    )

    return x, y

In [ ]:
def euclidean_distance_meters(latitude1, longitude1, latitude2, longitude2):
    """
       Parameters:
        lat1, lon1  -- New store coordinates
        lat2, lon2  -- CBG centroid coordinates
 
    #Returns:
        #Distance in meters (float)
    """
    x1, y1 = wgs84_to_utm_zone19n(latitude1, longitude1)
    x2, y2 = wgs84_to_utm_zone19n(latitude2, longitude2)
    return math.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)

print("\nCalculating distances to all CBG centroids")
print("Method: Euclidean distance via UTM Zone 19N projection (EPSG:32619)")


Calculating distances to all CBG centroids
Method: Euclidean distance via UTM Zone 19N projection (EPSG:32619)


In [ ]:
cbgs = cbgs.copy()
cbgs["dist_to_new"] = cbgs.apply(
    lambda r: euclidean_distance_meters(latitude_new, longitude_new, r["lat"], r["lon"]),
    axis = 1
)

# Avoid division by zero if new store is exactly at a centroid
cbgs["dist_to_new"] = cbgs["dist_to_new"].replace(0, 1.0)

print(f"Nearest CBG: {cbgs['dist_to_new'].min():,.0f} m")
print(f"Farthest CBG: {cbgs['dist_to_new'].max():,.0f} m")

Nearest CBG: 393 m
Farthest CBG: 8,081 m


In [ ]:
# ==============================================================
# The HUFF MODEL LOGIC
# ==============================================================
competitors = pois[
    (pois["top_category"].str.lower() == matched_category.lower()) |
    (pois["naics_code"].astype(str) == str(naics_code))
].copy()

print(f"Competitors in '{matched_category}': {len(competitors)}")

competitors_distance = distances[distances["placekey"].isin(competitors["placekey"])].copy()
competitors_distance = competitors_distance.merge(
    competitors[["placekey", "wkt_area_sq_meters"]],
    on = "placekey", how = "left"
)
median_size = competitors["wkt_area_sq_meters"].median()
competitors_distance["wkt_area_sq_meters"] = competitors_distance["wkt_area_sq_meters"].fillna(median_size)
competitors_distance["distance_n"] = competitors_distance["distance_m"].replace(0, 1.0)

# Competitor utility: U = area^alpha / distance^beta
competitors_distance["U_comp"] = (
    competitors_distance["wkt_area_sq_meters"] ** alpha / competitors_distance["distance_m"] ** beta
)

# Sum competitor utilities per CBG
cbg_sum_U = (
    competitors_distance.groupby("GEOID10")["U_comp"].sum().reset_index().rename(columns = {"GEOID10": "cbg", "U_comp": "sum_U_existing"})
)

cbgs = cbgs.merge(cbg_sum_U, on = "cbg", how = "left")
cbgs["sum_U_existing"] = cbgs["sum_U_existing"].fillna(0)

# New store utility per CBG
cbgs["U_new"] = (store_size ** alpha) / (cbgs["dist_to_new"] ** beta)

# Probability: P_new = U_new / (U_new + sum_U_existing)
total_U = cbgs["U_new"] + cbgs["sum_U_existing"]
cbgs["P_new"] = np.where(total_U > 0, cbgs["U_new"] / total_U, 0)

Competitors in 'Building Material and Supplies Dealers': 34


In [ ]:
# ==============================================================
# Demand Estimaiton
# ==============================================================
category_visits = visits[visits['placekey'].isin(competitors['placekey'])].copy()

cbg_demand = (
    category_visits.groupby("visitor_home_cbg")["visit_count"]
    .sum()
    .reset_index()
    .rename(columns = {"visitor_home_cbg": "cbg", "visit_count": "total_demand"})
)

cbgs = cbgs.merge(cbg_demand, on = "cbg", how = "left")
cbgs["total_demand"] = cbgs["total_demand"].fillna(0)
cbgs["predicted_visits"] = cbgs["P_new"] * cbgs["total_demand"]

In [ ]:
# ==============================================================
# Result
# ==============================================================
total_visits = cbgs["predicted_visits"].sum()
 
print("=" * 60)
print("PREDICTION RESULTS")
print("=" * 60)
print(f"\n  Location: ({latitude_new}, {longitude_new})")
print(f"Category: {matched_category}")
print(f"Store Size: {store_size:,.0f} sq meters")
print(f"Alpha|Beta: {alpha} | {beta}")
print(f"Competitors: {len(competitors)}")
print(f"\nTOTAL PREDICTED VISITS: {total_visits:,.1f}")
 
# Top 10 contributing neighborhoods
print("\nTop 10 Contributing Neighborhoods:")
print(f"{'CBG':<15} {'P_new':>8}  {'Demand':>8}  {'Predicted Visits':>16}")
 
top = (cbgs[cbgs["total_demand"] > 0]
       .sort_values("predicted_visits", ascending=False)
       .head(10))
 
for _, row in top.iterrows():
    print(f"  {int(row['cbg']):<15} {row['P_new']:>8.4f}  "
          f"{row['total_demand']:>8.0f}  {row['predicted_visits']:>16.1f}")


   PREDICTION RESULTS

  Location:    (42.2625, -71.8032)
  Category:    Building Material and Supplies Dealers
  Store Size:  2,500 sq meters
  Alpha|Beta:  1.9 | 1.0
  Competitors: 34

  >>> TOTAL PREDICTED VISITS: 25.5 <<<

  Top 10 Contributing Neighborhoods:
  CBG                P_new    Demand  Predicted Visits
  ----------------------------------------------------
  250277317001      0.0548       107               5.9
  250277326002      0.0239        96               2.3
  250277312032      0.0173        57               1.0
  250277315005      0.0274        34               0.9
  250277318003      0.0224        38               0.9
  250277313002      0.0239        33               0.8
  250277323012      0.0117        59               0.7
  250277323021      0.0152        41               0.6
  250277330001      0.0223        27               0.6
  250277324003      0.0267        22               0.6

   Script complete.
